# 08. staging 이미지 검수 및 승인 (→ processed)

**목적**: `data/staging/{CLASS_NAME}/`의 이미지를 검수해서 `data/processed/{CLASS_NAME}/`로 이동
(Cell 1의 `CLASS_NAME` 한 곳만 바꾸면 어떤 클래스에도 재사용 가능)

**실행 순서**
1. Cell 1 — 셋업 + 스테이징 스캔 (실제 파일 기준, 수동 삭제 자동 감지)
2. Cell 2 — HTML contact sheet 생성 (브라우저에서 검토)
3. Cell 3 — 노트북 인라인 그리드
4. **Cell 5** — `APPROVED` 목록에 파일 경로 기입 후 유효성 확인
5. Cell 6 — phash 중복 검사 + processed 이동 + 메타데이터 갱신
6. Cell 7 — 결과 확인 후 04 → 05 재실행

**이미지 삭제 방법**
- staging 폴더에서 **직접 파일을 삭제**하면 자동으로 감지됩니다
- 삭제된 파일은 metadata에 `deleted`로 기록되고 processed로 이동하지 않습니다
- APPROVED 목록에 없는 파일도 processed로 이동하지 않습니다

**APPROVED에 넣지 마세요**
- 제품 전체가 안 보이는 이미지 (partial crop)
- 내부 세제함·드럼 근접 촬영
- 설치 기사·광고 텍스트 위주
- 여러 제품이 겹쳐 주 제품 불명확한 이미지

In [ ]:
import os, sys, shutil, base64, re
from io import BytesIO
from pathlib import Path
from PIL import Image as PILImage
import imagehash
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
import config

CLASS_NAME   = 'wash_tower'
STAGING_DIR  = os.path.join('data', 'staging', CLASS_NAME)
PROC_DIR     = os.path.join(config.PROCESSED_DIR, CLASS_NAME)
os.makedirs(PROC_DIR, exist_ok=True)   # 새 클래스는 디렉토리가 없을 수 있음
METADATA_CSV = os.path.join(config.METADATA_DIR, f'{CLASS_NAME}_staging_metadata.csv')
PHASH_THRESH = 5

# ── ① 실제 파일 기준으로 staging 스캔 ─────────────────────────────────────────
staged = []   # [(slug, fpath)]
for slug in sorted(os.listdir(STAGING_DIR)):
    qd = os.path.join(STAGING_DIR, slug)
    if not os.path.isdir(qd) or slug.startswith('_'):
        continue
    for f in sorted(os.listdir(qd)):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            staged.append((slug, os.path.join(qd, f)))

staged_paths = {str(Path(fp)) for _, fp in staged}

by_q = {}
for slug, _ in staged:
    by_q[slug] = by_q.get(slug, 0) + 1

print(f'staging 실제 파일: {len(staged)}장')
for slug, cnt in sorted(by_q.items()):
    print(f'  {slug}: {cnt}장')

# ── ② metadata CSV와 교차 확인 — 수동 삭제 파일 감지 ──────────────────────────
n_manually_deleted = 0
if os.path.exists(METADATA_CSV):
    _meta = pd.read_csv(METADATA_CSV, encoding='utf-8-sig')
    _staged_rows = _meta[_meta['status'] == 'staged'].copy()
    _staged_rows['_norm'] = _staged_rows['saved_path'].apply(
        lambda p: str(Path(str(p))) if pd.notna(p) else ''
    )
    _missing_rows = _staged_rows[~_staged_rows['_norm'].isin(staged_paths)]
    n_manually_deleted = len(_missing_rows)
    if n_manually_deleted:
        print(f'\n[삭제 감지] metadata에는 있지만 실제 파일 없음: {n_manually_deleted}장')
        for _, row in _missing_rows.iterrows():
            print(f'  {row["saved_path"]}')
    else:
        print(f'\n[OK] metadata와 실제 파일 일치 (수동 삭제 없음)')
else:
    print(f'\n[INFO] metadata CSV 없음')

proc_imgs = [f for f in os.listdir(PROC_DIR)
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f'\n현재 processed/{CLASS_NAME}: {len(proc_imgs)}장')

In [ ]:
html_out = os.path.join(config.METADATA_DIR, f'{CLASS_NAME}_staging_review.html')

def to_b64(fpath):
    try:
        img = PILImage.open(fpath).convert('RGB')
        img.thumbnail((220,220), PILImage.LANCZOS)
        buf = BytesIO()
        img.save(buf, format='JPEG', quality=82)
        return base64.b64encode(buf.getvalue()).decode()
    except:
        return ''

CSS = ('<style>'
       '*{box-sizing:border-box}'
       'body{font-family:sans-serif;background:#111;color:#ccc;margin:0;padding:12px}'
       'h1{color:#fff;margin:0 0 4px}'
       '.stats{color:#888;font-size:13px;margin-bottom:16px}'
       '.qhead{color:#fa0;font-size:14px;font-weight:bold;margin:20px 0 8px;'
       'padding:4px 8px;background:#222;border-radius:4px}'
       '.grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(230px,1fr));gap:8px;margin-bottom:8px}'
       '.card{background:#222;border-radius:6px;padding:8px;position:relative}'
       '.card img{width:100%;height:200px;object-fit:contain;background:#0a0a0a;border-radius:4px;display:block}'
       '.noimg{width:100%;height:200px;background:#333;display:flex;align-items:center;justify-content:center;color:#555}'
       '.idx{position:absolute;top:4px;left:4px;background:rgba(0,0,0,.8);'
       'color:#fff;padding:1px 6px;border-radius:3px;font-size:11px}'
       '.fname{font-size:10px;margin-top:5px;word-break:break-all;color:#999}'
       '.slug{font-size:10px;color:#555;margin-top:2px}'
       '</style>')

sections = []
global_i = 0
for slug in sorted(by_q.keys()):
    cards = []
    for slug2, fpath in staged:
        if slug2 != slug:
            continue
        fname  = os.path.basename(fpath)
        b64    = to_b64(fpath)
        itag   = (f'<img src="data:image/jpeg;base64,{b64}" loading="lazy">'
                  if b64 else '<div class="noimg">ERR</div>')
        relpath = fpath.replace(os.sep, '/')
        cards.append(
            f'<div class="card">'
            f'<div class="idx">#{global_i}</div>'
            f'{itag}'
            f'<div class="fname">{fname}</div>'
            f'<div class="slug">{relpath}</div>'
            f'</div>'
        )
        global_i += 1
    sections.append(
        f'<div class="qhead">{slug} ({len(cards)}장)</div>'
        f'<div class="grid">' + ''.join(cards) + '</div>'
    )

html = (
    '<!DOCTYPE html><html><head><meta charset="utf-8">'
    f'<title>staging review</title>' + CSS + '</head><body>'
    f'<h1>{CLASS_NAME} 스테이징 검수 — {len(staged)}장</h1>'
    f'<div class="stats">APPROVED 목록에 넣을 파일의 전체 경로(saved_path)를 복사하세요.</div>'
    + ''.join(sections) + '</body></html>'
)
with open(html_out, 'w', encoding='utf-8') as f:
    f.write(html)
print(f'HTML 저장: {html_out}')

In [ ]:
BATCH = 30
N_COLS = 5
all_paths = [(slug, fpath) for slug, fpath in staged]

for start in range(0, len(all_paths), BATCH):
    batch = all_paths[start:start+BATCH]
    n_rows = (len(batch)+N_COLS-1)//N_COLS
    fig, axes = plt.subplots(n_rows, N_COLS, figsize=(20, 4*n_rows))
    axes = np.array(axes).reshape(-1) if n_rows*N_COLS > 1 else [axes]
    for ax_i,(slug,fpath) in enumerate(batch):
        idx = start+ax_i
        try:
            img = PILImage.open(fpath).convert('RGB')
        except:
            axes[ax_i].text(0.5,0.5,'ERR',ha='center',va='center',
                            transform=axes[ax_i].transAxes)
            axes[ax_i].axis('off')
            continue
        axes[ax_i].imshow(img)
        short = os.path.basename(fpath)
        axes[ax_i].set_title(f'#{idx}\n{slug[:15]}\n{short}',fontsize=6)
        axes[ax_i].axis('off')
    for ax_i in range(len(batch),len(axes)):
        axes[ax_i].axis('off')
    end = min(start+BATCH-1,len(all_paths)-1)
    plt.suptitle(f'staging #{start}~#{end}',fontsize=11,fontweight='bold')
    plt.tight_layout()
    plt.show()

## 검수 기준

**APPROVED에 추가할 이미지** — 아래 기준을 모두 충족
- 세탁기·건조기 **제품 전체** 외관이 보임
- 배경이 복잡해도 주 제품이 명확히 식별 가능
- 해상도 150px 이상 (자동 필터됨)

**APPROVED에 넣지 않을 이미지**
- 제품 일부(모서리, 조작판만)만 잘린 이미지
- 내부 드럼·세제함 근접 촬영
- 설치 기사, 광고 텍스트 위주
- 여러 제품이 겹쳐 주 제품 불명확

Cell 5의 `APPROVED` 리스트에 `saved_path` 값을 붙여넣으세요.  
HTML의 이미지 카드 하단에 전체 경로가 표시됩니다.

In [ ]:
# ── APPROVED: processed로 이동할 파일 전체 경로 ───────────────────────────────
# HTML contact sheet의 카드 하단 경로를 복사해서 붙여넣으세요
# 슬래시(/) 또는 역슬래시(\) 모두 허용됩니다
# 경로 예시: {STAGING_DIR}/쿼리폴더명/stg_0001.jpg

APPROVED = [
    # f'{STAGING_DIR}/쿼리폴더명/stg_0000.jpg',
    # f'{STAGING_DIR}/쿼리폴더명/stg_0002.jpg',
]

# 경로 구분자를 OS 기준으로 통일 + 실제 존재 여부 확인
approved_norm    = [str(Path(p)) for p in APPROVED]
approved_exist   = [p for p in approved_norm if os.path.exists(p)]
approved_missing = [p for p in approved_norm if not os.path.exists(p)]

print(f'APPROVED 목록: {len(APPROVED)}장')
print(f'  파일 존재:  {len(approved_exist)}장  -> Cell 6에서 처리 예정')
if approved_missing:
    print(f'  파일 없음:  {len(approved_missing)}장  -> 자동 제외 (수동 삭제로 처리)')
    for p in approved_missing:
        print(f'    {p}')
else:
    print(f'  파일 없음:  0장')

In [ ]:
# ── phash 중복 검사 → processed 이동 ───────────────────────────────────────
# approved_exist (Cell 5에서 존재 확인된 목록)만 처리
# APPROVED에 없거나 파일이 이미 삭제된 항목은 processed로 이동하지 않음

print('기존 processed phash 계산 중...')
proc_hashes = {}
for f in tqdm(os.listdir(PROC_DIR), desc='processed phash', leave=False):
    if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    try:
        proc_hashes[f] = imagehash.phash(PILImage.open(os.path.join(PROC_DIR, f)))
    except Exception:
        pass

naver_nums = [
    int(f[6:-4]) for f in os.listdir(PROC_DIR)
    if f.startswith('naver_') and f[6:-4].isdigit()
]
next_idx = (max(naver_nums) + 1) if naver_nums else 0

metadata_df = (pd.read_csv(METADATA_CSV, encoding='utf-8-sig') if os.path.exists(METADATA_CSV)
               else pd.DataFrame())

moved, dedup_skip = [], []

for src_path in approved_exist:
    try:
        h = imagehash.phash(PILImage.open(src_path))
    except Exception as e:
        print(f'[오류] {os.path.basename(src_path)}: {e}')
        continue

    dup = [(f, h - ph) for f, ph in proc_hashes.items() if h - ph <= PHASH_THRESH]
    if dup:
        dup.sort(key=lambda x: x[1])
        dedup_skip.append((src_path, dup[0]))
        print(f'[중복] {os.path.basename(src_path)} ~= {dup[0][0]} (거리 {dup[0][1]})')
        continue

    new_name = f'naver_{next_idx:04d}.jpg'
    dst_path = os.path.join(PROC_DIR, new_name)
    shutil.move(src_path, dst_path)
    proc_hashes[new_name] = h   # 동일 배치 내 중복 방지

    moved.append((src_path, dst_path))
    print(f'이동: {os.path.basename(src_path)}  ->  {new_name}')
    next_idx += 1

# ── metadata 갱신 ─────────────────────────────────────────────────────────────
if not metadata_df.empty and 'saved_path' in metadata_df.columns:

    def _norm(p):
        try:
            return str(Path(str(p))) if pd.notna(p) else ''
        except Exception:
            return ''

    norm_col = metadata_df['saved_path'].apply(_norm)

    # 이동 성공한 파일 → approved (saved_path도 새 경로로 갱신)
    for src, dst in moved:
        mask = norm_col == _norm(src)
        metadata_df.loc[mask, 'status']     = 'approved'
        metadata_df.loc[mask, 'saved_path'] = dst

    # APPROVED에 있었지만 파일이 없던 경로 → missing
    for p in approved_missing:
        mask = norm_col == _norm(p)
        metadata_df.loc[mask, 'status'] = 'missing'

    # metadata 기준 staged인데 실제 파일이 없는 항목 → deleted (수동 삭제)
    staged_mask    = metadata_df['status'] == 'staged'
    norm_col_fresh = metadata_df['saved_path'].apply(_norm)
    deleted_mask   = staged_mask & ~norm_col_fresh.isin(staged_paths)
    n_deleted      = int(deleted_mask.sum())
    metadata_df.loc[deleted_mask, 'status'] = 'deleted'

    metadata_df.to_csv(METADATA_CSV, index=False, encoding='utf-8-sig')
    print(f'\nmetadata 갱신: {METADATA_CSV}')
    if n_deleted:
        print(f'  수동 삭제 처리: {n_deleted}건 (status=deleted)')

print(f'\n결과 -- 이동 {len(moved)}장 | 중복 제외 {len(dedup_skip)}장 | APPROVED 파일 없음 {len(approved_missing)}장')

In [ ]:
# ── 최종 현황 ─────────────────────────────────────────────────────────────────
print('=== processed/ 현황 ===')
for cls in sorted(os.listdir(config.PROCESSED_DIR)):
    p = os.path.join(config.PROCESSED_DIR, cls)
    if not os.path.isdir(p):
        continue
    imgs = [f for f in os.listdir(p) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if not imgs:
        continue
    oi_n = len([f for f in imgs if not f.startswith('naver_')])
    nv_n = len([f for f in imgs if f.startswith('naver_')])
    print(f'  {cls}: {len(imgs)}장  (OI {oi_n} / Naver {nv_n})')

# staging 잔여 파일 재스캔 (실제 파일 기준)
remain_staged = []
for slug in sorted(os.listdir(STAGING_DIR)):
    d = os.path.join(STAGING_DIR, slug)
    if not os.path.isdir(d) or slug.startswith('_'):
        continue
    for f in os.listdir(d):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            remain_staged.append(os.path.join(d, f))

# metadata 상태별 집계
status_counts = {}
if os.path.exists(METADATA_CSV):
    mdf = pd.read_csv(METADATA_CSV, encoding='utf-8-sig')
    status_counts = mdf['status'].value_counts().to_dict()

print()
print('=== 이번 세션 결과 ===')
print(f'  승인 -> processed 이동: {len(moved)}장')
print(f'  phash 중복 제외:        {len(dedup_skip)}장')
if approved_missing:
    print(f'  APPROVED 파일 없음:     {len(approved_missing)}장 (자동 제외)')

print()
print('=== staging 잔여 ===')
print(f'  실제 남은 파일: {len(remain_staged)}장')

print()
print('=== metadata 상태 집계 ===')
for st in ['approved', 'staged', 'deleted', 'missing']:
    cnt = status_counts.get(st, 0)
    if cnt:
        print(f'  {st:10s}: {cnt}건')

print()
print('=== 다음 단계 ===')
if len(moved) > 0:
    print('1. 04_split_dataset.ipynb 전체 재실행  (split 재생성)')
    print('2. 05_train_efficientnetv2.ipynb 처음부터 재실행  (재학습)')
else:
    print('승인된 이미지가 없습니다.')
    print('APPROVED 목록을 채우고 Cell 6을 다시 실행하세요.')